In [1]:
"""
Phoneme boundary evaluation -- Rhapsodie corpus
Model: w2vCTC_bertphone  (Wav2Vec2 + XPhoneBERT + Forward-Sum alignment)

Architecture exactly matches training:
  - Wav2Vec2 encoder  -> X  (B, T, 1024)
  - CTC head          -> logits -> greedy decode -> labels_clean
  - YOUR_TO_IPA       -> XPhoneBERT (frozen) -> bert_proj  -> Y_emb (B, N, 1024)
  - align_temperature = 15.0, prior = gaussian(-30) + ahead_penalty(-40)
  - A = softmax(15 * cosine(Y_emb, fx(X)) + prior, dim=1)

Diagnostics:
  1. Boundary accuracy  @ +/-10 / 20 / 50 ms
  2. Boundary error histogram
  3. Positional bias (running-ahead diagnosis)
  4. Alignment matrix quality (sharpness, diagonal coverage, entropy)
  5. Per-phoneme boundary accuracy
  6. Duration vs boundary accuracy
"""

# == Imports ==================================================================
import os
import glob
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import soundfile as sf
import tgt
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from collections import defaultdict
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    AutoTokenizer,
    AutoModel,
)

os.environ["CUDA_VISIBLE_DEVICES"] = "1"


# =============================================================================
# CTC vocab -> IPA mapping  (must match training exactly)
# =============================================================================

YOUR_TO_IPA = {
    "a":  "a",  "b": "b",  "d": "d",  "e": "e",  "f": "f",
    "i":  "i",  "j": "j",  "k": "k",  "l": "l",  "m": "m",
    "n":  "n",  "o": "o",  "p": "p",  "s": "s",  "t": "t",
    "u":  "u",  "v": "v",  "w": "w",  "y": "y",  "z": "z",
    "ø":  "ø",  "ŋ": "ŋ",  "ɔ": "ɔ",  "ə": "ə",  "ɛ": "ɛ",
    "ɡ":  "ɡ",  "ɲ": "ɲ",  "ʁ": "ʁ",  "ʃ": "ʃ",  "ʒ": "ʒ",
    # collapses
    "dʒ": "ʒ", "tʃ": "ʃ", "ts": "s", "@": "ə",
    # silence / special -> skip
    "§": None, "*": None, "[PAD]": None, "[UNK]": None,
    # numeric tokens in vocab
    "1": None, "5": None, "9": None,
}


# =============================================================================
# Model
# =============================================================================

class GumbelQuantizerEMA(nn.Module):
    def __init__(self, input_dim: int, num_vars: int = 320,
                 temp: float = 2.0, decay: float = 0.99):
        super().__init__()
        self.num_vars = num_vars
        self.temp     = temp
        self.decay    = decay
        self.weight_proj = nn.Linear(input_dim, num_vars)
        self.register_buffer("codevectors", torch.randn(num_vars, input_dim))
        self.register_buffer("cluster_size", torch.ones(num_vars))
        self.register_buffer("ema_embed",   torch.randn(num_vars, input_dim))
        nn.init.uniform_(self.codevectors, -1.0, 1.0)

    def forward(self, X: torch.Tensor):
        B, T, D = X.shape
        logits  = self.weight_proj(X)
        if self.training:
            probs = F.gumbel_softmax(logits, tau=self.temp, hard=True)
            with torch.no_grad():
                flat_probs = probs.reshape(-1, self.num_vars)
                flat_X     = X.reshape(-1, D)
                counts     = flat_probs.sum(0)
                embed_sum  = flat_probs.T @ flat_X
                self.cluster_size = (self.decay * self.cluster_size
                                     + (1 - self.decay) * counts)
                self.ema_embed    = (self.decay * self.ema_embed
                                     + (1 - self.decay) * embed_sum)
                self.codevectors  = (self.ema_embed
                                     / self.cluster_size.unsqueeze(1).clamp(min=1e-5))
        else:
            indices = logits.argmax(dim=-1)
            probs   = F.one_hot(indices, self.num_vars).float()
        return torch.matmul(probs, self.codevectors), probs


class Wav2Vec2ForCTC_FS_REC(nn.Module):
    def __init__(self, base_model, vocab_size, hidden_dim,
                 ctc_tokenizer, bert_tokenizer,
                 n_negatives=50, temperature=0.1):
        super().__init__()
        self.wav2vec2      = base_model.wav2vec2
        self.ctc_head      = base_model.lm_head
        self.ctc_tokenizer = ctc_tokenizer
        self.bert_tokenizer = bert_tokenizer
        self.n_negatives   = n_negatives
        self.temperature   = temperature

        # Alignment hyper-parameters -- must match training exactly
        self.align_temperature = 15.0
        self.gaussian_weight   = 30.0
        self.ahead_weight      = 40.0

        # Trainable modules
        self.fx              = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.bert_proj       = nn.Sequential(
            nn.Linear(768, hidden_dim),
            nn.ReLU(),
        )
        self.quantizer           = GumbelQuantizerEMA(input_dim=hidden_dim)
        self.reconstruction_head = nn.Linear(2 * hidden_dim, hidden_dim)

        # XPhoneBERT -- frozen at inference (as in training)
        self.phoneme_bert = AutoModel.from_pretrained("vinai/xphonebert-base")
        for p in self.phoneme_bert.parameters():
            p.requires_grad = False

    # -------------------------------------------------------------------------
    def _ctc_decode_batch(self, logits):
        pred_ids  = logits.argmax(dim=-1)
        sequences = []
        for seq in pred_ids:
            unique_mask = torch.cat([
                torch.tensor([True], device=seq.device),
                seq[1:] != seq[:-1],
            ])
            collapsed = seq[unique_mask]
            collapsed = collapsed[collapsed != 0]
            if collapsed.numel() == 0:
                collapsed = torch.tensor([1], device=seq.device)
            sequences.append(collapsed)
        return torch.nn.utils.rnn.pad_sequence(
            sequences, batch_first=True, padding_value=0
        )

    # -------------------------------------------------------------------------
    def compute_phoneme_embeddings(self, labels_clean, device):
        """
        labels_clean (B, N_pad) -- CTC token ids (0 = PAD)
        Returns Y_emb (B, N_max, hidden_dim) in encoder space.

        Pipeline per sample:
          token_id -> CTC token string -> YOUR_TO_IPA -> IPA char
          -> space-joined IPA sequence -> XPhoneBERT -> bert_proj

        For XPhoneBERT: each input phoneme is a single space-separated token.
        We recover per-phoneme embeddings by mapping XPhoneBERT subword pieces
        back to their source phoneme (first-subword strategy, matching training).
        """
        B, N_pad = labels_clean.shape
        Y_emb_list = []

        for b in range(B):
            ids = labels_clean[b]
            # Resolve token strings and IPA, skipping PAD
            ipa_seq  = []   # IPA string per phoneme (or None = skip)
            keep_idx = []   # original position in ids
            for i, tok_id in enumerate(ids.tolist()):
                if tok_id == 0:     # PAD
                    continue
                tok_str = self.ctc_tokenizer.convert_ids_to_tokens(tok_id)
                ipa     = YOUR_TO_IPA.get(tok_str)
                if ipa is not None:
                    ipa_seq.append(ipa)
                    keep_idx.append(i)

            N = len(ipa_seq)
            hidden_dim = self.bert_proj[0].out_features

            if N == 0:
                Y_emb_list.append(torch.zeros(1, hidden_dim, device=device))
                continue

            # Tokenise space-joined IPA sequence
            ipa_str = " ".join(ipa_seq)
            enc = self.bert_tokenizer(
                ipa_str,
                return_tensors        = "pt",
                return_offsets_mapping = False,
                add_special_tokens    = True,   # [CLS] ... [SEP]
            )
            enc = {k: v.to(device) for k, v in enc.items()}

            with torch.no_grad():
                bert_out = self.phoneme_bert(**enc)
            # last_hidden_state: (1, L, 768)  L includes [CLS] and [SEP]
            hidden = bert_out.last_hidden_state[0]  # (L, 768)

            # Map subword pieces back to phonemes.
            # XPhoneBERT's sentencepiece tokeniser splits on the spaces we
            # inserted, so each IPA character becomes >=1 piece.
            # We use the first-subword embedding (consistent with training).
            piece_ids = enc["input_ids"][0].tolist()      # (L,)
            pieces    = self.bert_tokenizer.convert_ids_to_tokens(piece_ids)

            # Walk through pieces and collect the first non-special piece
            # that belongs to each phoneme.
            special = {
                self.bert_tokenizer.cls_token,
                self.bert_tokenizer.sep_token,
                self.bert_tokenizer.pad_token,
            }
            ph_embs  = []
            ph_idx   = 0       # which IPA phoneme we are building
            in_ph    = False   # are we past the first piece of current phoneme?

            for pos, piece in enumerate(pieces):
                if piece in special:
                    continue
                # Sentencepiece: pieces belonging to the *same* word do NOT
                # start with the special continuation marker (they lack the
                # leading space / U+2581 prefix).
                is_word_start = piece.startswith("\u2581") or not in_ph

                if is_word_start:
                    if ph_idx < N:
                        ph_embs.append(hidden[pos])   # first-piece embedding
                        ph_idx  += 1
                        in_ph    = True
                # subsequent pieces of same phoneme: ignored (first-subword)

            # Safety: if some phonemes got no piece (very rare edge case),
            # pad with zeros so the tensor stays (N, 768).
            while len(ph_embs) < N:
                ph_embs.append(torch.zeros(768, device=device))

            Y_raw = torch.stack(ph_embs[:N])           # (N, 768)
            Y_proj = self.bert_proj(Y_raw)              # (N, hidden_dim)
            Y_emb_list.append(Y_proj)

        # Pad to uniform N across batch
        Y_emb = torch.nn.utils.rnn.pad_sequence(
            Y_emb_list, batch_first=True, padding_value=0.0
        )  # (B, N_max, hidden_dim)
        return Y_emb

    # -------------------------------------------------------------------------
    def compute_alignment(self, X, Y_emb):
        """
        Exact training formula:
          D = 15 * normalize(Y_emb @ fx(X).T)
          prior = gaussian(-30) + ahead_penalty(-40)
          A = softmax(D + prior, dim=1)
        """
        B, T, D = X.shape
        _, N, _ = Y_emb.shape

        X_proj = self.fx(X)
        X_norm = F.normalize(X_proj, dim=-1)
        Y_norm = F.normalize(Y_emb,  dim=-1)
        D_mat  = self.align_temperature * torch.matmul(
            Y_norm, X_norm.transpose(1, 2)
        )   # (B, N, T)

        n_idx = torch.arange(N, device=X.device).float() / max(N - 1, 1)
        t_idx = torch.arange(T, device=X.device).float() / max(T - 1, 1)
        diff  = n_idx.unsqueeze(1) - t_idx.unsqueeze(0)   # (N, T)

        gaussian      = -self.gaussian_weight * diff ** 2
        ahead_penalty = -self.ahead_weight    * torch.clamp(diff, min=0) ** 2
        prior         = gaussian + ahead_penalty                           # (N, T)

        return torch.softmax(D_mat + prior.unsqueeze(0), dim=1)   # (B, N, T)

    # -------------------------------------------------------------------------
    def forward(self, input_values, attention_mask=None):
        outputs = self.wav2vec2(input_values, attention_mask=attention_mask)
        X       = outputs.last_hidden_state                    # (B, T, 1024)
        logits  = self.ctc_head(X)

        with torch.no_grad():
            labels_clean = self._ctc_decode_batch(logits)

        Y_emb = self.compute_phoneme_embeddings(labels_clean, device=X.device)
        A     = self.compute_alignment(X, Y_emb)
        return logits, A, labels_clean


# =============================================================================
# Label mappings
# =============================================================================

# SAMPA (TextGrid) -> model vocab.  None = silence / skip.
sampa_to_model = {
    'a': 'a',  'e': 'e',  'i': 'i',  'o': 'o',  'u': 'u',  'y': 'y',
    '2': 'ø',  '9': '9',  '@': 'ə',  'E': 'ɛ',  'O': 'ɔ',
    'a~': '@', 'e~': '5', '9~': '1', 'o~': '§',
    'b': 'b',  'd': 'd',  'f': 'f',  'g': 'ɡ',  'k': 'k',  'l': 'l',
    'm': 'm',  'n': 'n',  'n=': 'n', 'p': 'p',  't': 't',  'v': 'v',
    'w': 'w',  'z': 'z',  'j': 'j',  's': 's',  'm=': 'm',
    'R': 'ʁ',  'N': 'ŋ',
    'H': None,   # ɥ not in 40-token vocab
    'J': 'ɲ',  'S': 'ʃ',  'Z': 'ʒ',  'Z=': 'dʒ',
    '_': None, 'spn': None, 'unk': None, '%': None, '?': None, '0': None,
}

# model tokenizer output -> ref label space
hyp_to_ref = {
    **sampa_to_model,
    '@': '@',    # vocab '@' = nasal a~ not schwa
    'ts': 's',
    'tʃ': 'ʃ',
}

SILENCE_SET = {
    "", "SIL", "sil", "spn", "SP", "<SIL>", "_", "0",
    "fe~", "sjo~", "Ra~", None,
}


def normalise_hyp_token(tok: str):
    return hyp_to_ref.get(tok, tok)


def map_ref_to_model(ph: str):
    return sampa_to_model.get(ph, ph if (ph and ph != "") else None)


# =============================================================================
# TextGrid reader
# =============================================================================

def read_textgrid(tg_path: str, tier_name: str = "phone") -> list:
    tg   = tgt.io.read_textgrid(tg_path)
    tier = tg.get_tier_by_name(tier_name)
    out  = []
    for iv in tier.intervals:
        label   = iv.text.strip()
        phoneme = map_ref_to_model(label)
        if phoneme in SILENCE_SET:
            continue
        out.append({
            "phoneme": phoneme,
            "start":   round(iv.start_time, 6),
            "end":     round(iv.end_time,   6),
        })
    return out


def get_ref_intervals(clean_ref: list, audio_path: str,
                      threshold: float = 0.5) -> list:
    """Subtract session-level time offset when TextGrid timestamps exceed clip duration."""
    if not clean_ref:
        return []
    audio, sr      = sf.read(audio_path)
    audio_duration = len(audio) / sr
    ref_last       = clean_ref[-1]["end"]
    if ref_last > audio_duration + threshold:
        ref_offset = clean_ref[0]["start"]
        print(f"  Session offset: ref_last={ref_last:.2f}s "
              f"audio={audio_duration:.2f}s offset={ref_offset:.3f}s")
        return [
            {
                "phoneme": iv["phoneme"],
                "start":   max(0.0, round(iv["start"] - ref_offset, 6)),
                "end":     max(0.0, round(iv["end"]   - ref_offset, 6)),
            }
            for iv in clean_ref
        ]
    return list(clean_ref)


# =============================================================================
# Boundary extraction from alignment matrix
# =============================================================================

def _extract_boundaries_from_A(A_np: np.ndarray, N: int, T: int) -> list:
    """Monotonic frame boundaries via running-max on column argmax. O(T)."""
    if N == 0:
        return [0, T]
    if T < N:
        return list(range(N + 1))
    assignments = A_np.argmax(axis=0)
    for t in range(1, T):
        if assignments[t] < assignments[t - 1]:
            assignments[t] = assignments[t - 1]
    boundaries = [0]
    for n in range(1, N):
        frames = np.where(assignments == n)[0]
        boundaries.append(
            int(frames[0]) if len(frames) > 0
            else min(boundaries[-1] + 1, T - (N - n))
        )
    boundaries.append(T)
    return boundaries


def extract_intervals_forced(logits, A, tokenizer, duration_sec: float,
                              frame_shift: float = 0.02) -> list:
    blank_id = tokenizer.pad_token_id
    pred_ids = logits[0].argmax(dim=-1).tolist()

    collapsed, collapsed_pos = [], []
    prev = None
    for t, tok in enumerate(pred_ids):
        if tok != prev:
            if tok != blank_id:
                collapsed.append(tok)
                collapsed_pos.append(t)
        prev = tok

    if not collapsed:
        return []

    N    = len(collapsed)
    A_np = A[0, :N, :].cpu().float().numpy()
    T    = A_np.shape[1]

    # Speech onset (only applied to first interval start)
    min_run = 2
    run, first_speech_frame = 0, collapsed_pos[0]
    for t, tok in enumerate(pred_ids):
        if tok != blank_id:
            run += 1
            if run >= min_run:
                first_speech_frame = t - (min_run - 1)
                break
        else:
            run = 0

    boundaries = _extract_boundaries_from_A(A_np, N, T)

    skip = {"[PAD]", "[UNK]", "spn", "", None}
    intervals = []
    for n in range(N):
        raw_tok     = tokenizer.convert_ids_to_tokens(collapsed[n])
        phoneme_str = normalise_hyp_token(raw_tok)
        if phoneme_str not in skip:
            intervals.append({
                "phoneme": phoneme_str,
                "start":   round(boundaries[n]     * frame_shift, 6),
                "end":     round(boundaries[n + 1] * frame_shift, 6),
            })

    if intervals:
        onset = round(first_speech_frame * frame_shift, 6)
        intervals[0]["start"] = min(onset, intervals[0]["end"] - frame_shift)

    return intervals






In [7]:
# =============================================================================
# Config
# =============================================================================

checkpoint     = "results/w2vCTC_bertphone/checkpoint-14841"
ctc_checkpoint = "results/w2vCTC/checkpoint-23430"
audio_dir      = "/vol/corpora/Rhapsodie/wav16k_corrected/"
textgrid_dir   = "/vol/corpora/Rhapsodie/TextGrids-fev2013/"
vocab          = "vocab_w2vCTC.json"
output         = "results/rhap_bertphone.pkl"
tier           = "phone"
device         = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("results", exist_ok=True)
print(f"Device: {device}")


# =============================================================================
# Processor + model
# =============================================================================

print("Loading CTC tokenizer...")
ctc_tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file           = vocab,
    unk_token            = "[UNK]",
    pad_token            = "[PAD]",
    word_delimiter_token = "",
)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size          = 1,
    sampling_rate         = 16000,
    padding_value         = 0.0,
    do_normalize          = True,
    return_attention_mask = True,
)

print("Loading XPhoneBERT tokenizer...")
bert_tokenizer = AutoTokenizer.from_pretrained("vinai/xphonebert-base")

print(f"Loading Wav2Vec2 base from {ctc_checkpoint}...")
base_model = Wav2Vec2ForCTC.from_pretrained(
    ctc_checkpoint,
    ctc_loss_reduction = "mean",
    ctc_zero_infinity  = True,
    pad_token_id       = ctc_tokenizer.pad_token_id,
    vocab_size         = len(ctc_tokenizer),
)

print("Building joint model...")
model = Wav2Vec2ForCTC_FS_REC(
    base_model,
    vocab_size      = len(ctc_tokenizer),
    hidden_dim      = base_model.config.hidden_size,
    ctc_tokenizer   = ctc_tokenizer,
    bert_tokenizer  = bert_tokenizer,
)
model.wav2vec2.feature_extractor._freeze_parameters()

print(f"Loading joint weights from {checkpoint}...")
bin_path = os.path.join(checkpoint, "pytorch_model.bin")
sft_path = os.path.join(checkpoint, "model.safetensors")
if os.path.exists(bin_path):
    state_dict = torch.load(bin_path, map_location="cpu")
elif os.path.exists(sft_path):
    from safetensors.torch import load_file
    state_dict = load_file(sft_path)
else:
    raise FileNotFoundError(
        f"No weights found in {checkpoint}.\n"
        f"Expected pytorch_model.bin or model.safetensors"
    )

missing, unexpected = model.load_state_dict(state_dict, strict=False)
# phoneme_bert weights are loaded separately from HuggingFace (frozen),
# so they will appear in 'unexpected' if saved inside the checkpoint -- that's fine.
# What you do NOT want to see here: fx, bert_proj, ctc_head in missing.
critical = [k for k in missing if any(
    k.startswith(m) for m in ("fx.", "bert_proj.", "ctc_head.", "wav2vec2.")
)]
if critical:
    print(f"  WARNING -- critical keys missing: {critical[:10]}")
else:
    print(f"  OK -- no critical keys missing "
          f"({len(missing)} missing total, likely phoneme_bert overlap)")

model.eval()
model.to(device)
print("Model ready.\n")






Device: cuda
Loading CTC tokenizer...
Loading XPhoneBERT tokenizer...
Loading Wav2Vec2 base from results/w2vCTC/checkpoint-23430...


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Building joint model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/xphonebert-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading joint weights from results/w2vCTC_bertphone/checkpoint-14841...
  OK -- no critical keys missing (0 missing total, likely phoneme_bert overlap)
Model ready.



In [12]:
from VAD_chunk import *
results={}
n_done=0
for audio_path in os.listdir(audio_dir):
    audio_path = audio_dir+audio_path
    filename = os.path.splitext(os.path.basename(audio_path))[0]

    if filename == "Rhap-D2004":
        continue

    tg_path = os.path.join(textgrid_dir, filename + "-Pro.TextGrid")
    if not os.path.exists(tg_path):
        print(f"  [SKIP] No TextGrid for {filename}")
        continue

    clean_ref     = read_textgrid(tg_path, tier)
    ref_intervals = get_ref_intervals(clean_ref, audio_path)

    audio, sr = sf.read(audio_path)
    if sr != 16000:
        import librosa
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    wav    = torch.from_numpy(audio.astype(np.float32))
    chunks = vad_chunk_with_timestamps(wav)
    if not chunks:
        chunks = [{"start": 0.0, "end": len(audio) / 16000}]

    # ── Per-chunk inference ───────────────────────────────────────────
    hyp_intervals = []
    a_matrices    = []
    all_logits    = []   # for PER: concatenate CTC logits across chunks
    
    for chunk in chunks:
        start_sample = int(chunk["start"] * 16000)
        end_sample   = int(chunk["end"]   * 16000)
        chunk_audio  = audio[start_sample:end_sample]

        if len(chunk_audio) < 400:   # < 25ms — too short for wav2vec2
            continue

        inputs = feature_extractor(
            chunk_audio, sampling_rate=16000,
            return_tensors="pt", return_attention_mask=True,
        )
        with torch.no_grad():
            logits, A, _ = model(
                inputs.input_values.to(device),
                inputs.attention_mask.to(device),
            )

        dur = len(chunk_audio) / 16000.0
        ivs = extract_intervals_forced(logits, A, ctc_tokenizer, dur)

        # Shift intervals to absolute file time
        offset = chunk["start"]
        for iv in ivs:
            hyp_intervals.append({
                "phoneme": iv["phoneme"],
                "start":   round(iv["start"] + offset, 6),
                "end":     round(iv["end"]   + offset, 6),
            })

        # Store A and logits
        N = len(ivs)
        if N > 0:
            a_matrices.append(A[0, :N, :].cpu())
        all_logits.append(logits.cpu())

    results[filename] = {
        "file":          os.path.basename(audio_path),
        "ref_intervals": ref_intervals,
        "hyp_intervals": hyp_intervals,
        "A_chunks":      a_matrices,
        "logits_chunks": all_logits,   # list of per-chunk logits for PER
    }

    n_done += 1
    if n_done % 10 == 0 or n_done == 1:
        print(f"  [{n_done}/{len(audio_dir)}] {filename}: "
              f"ref={len(ref_intervals)} hyp={len(hyp_intervals)} "
              f"chunks={len(chunks)}")

  Session offset: ref_last=17.72s audio=17.02s offset=0.707s
  [1/40] Rhap-M0012: ref=165 hyp=164 chunks=1
  Session offset: ref_last=431.89s audio=427.88s offset=4.010s
  Session offset: ref_last=101.02s audio=100.44s offset=0.573s
  Session offset: ref_last=313.07s audio=310.95s offset=2.114s
  Session offset: ref_last=202.67s audio=201.18s offset=1.495s
  Session offset: ref_last=508.66s audio=499.39s offset=9.264s
  Session offset: ref_last=17.80s audio=17.07s offset=0.726s
  [10/40] Rhap-M0002: ref=526 hyp=514 chunks=4
  Session offset: ref_last=293.34s audio=292.68s offset=0.661s
  Session offset: ref_last=240.88s audio=239.41s offset=1.468s
  Session offset: ref_last=63.66s audio=62.52s offset=1.138s
  Session offset: ref_last=99.46s audio=98.29s offset=1.164s
  Session offset: ref_last=308.05s audio=306.80s offset=1.246s
  [20/40] Rhap-M1003: ref=2513 hyp=2485 chunks=16
  Session offset: ref_last=42.24s audio=41.66s offset=0.575s
  Session offset: ref_last=88.09s audio=87.39s o

In [13]:
def edit_distance(ref: list, hyp: list):
    """Standard dynamic programming edit distance. Returns (S, D, I)."""
    R, H = len(ref), len(hyp)
    dp = np.zeros((R + 1, H + 1), dtype=int)
    dp[:, 0] = np.arange(R + 1)
    dp[0, :] = np.arange(H + 1)
    for i in range(1, R + 1):
        for j in range(1, H + 1):
            if ref[i-1] == hyp[j-1]:
                dp[i, j] = dp[i-1, j-1]
            else:
                dp[i, j] = 1 + min(dp[i-1, j-1],  # substitution
                                    dp[i-1, j],     # deletion
                                    dp[i,   j-1])   # insertion
    # Backtrace to count S / D / I
    i, j = R, H
    S = D = I = 0
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref[i-1] == hyp[j-1]:
            i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            S += 1; i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            D += 1; i -= 1
        else:
            I += 1; j -= 1
    return S, D, I


def compute_per(results, ctc_tokenizer):
    total_S = total_D = total_I = total_ref = 0
    per_file = {}
    blank_id = ctc_tokenizer.pad_token_id
    skip     = {"[PAD]", "[UNK]", "spn", "", None}

    for fname, entry in results.items():
        ref_seq = [iv["phoneme"] for iv in entry["ref_intervals"]]
        if not ref_seq:
            continue

        # Reconstruct hyp sequence from per-chunk logits
        hyp_seq = []
        for logits in entry["logits_chunks"]:
            pred_ids = logits[0].argmax(dim=-1).tolist()
            prev = None
            for tok in pred_ids:
                if tok != prev:
                    if tok != blank_id:
                        raw = ctc_tokenizer.convert_ids_to_tokens(tok)
                        ph  = normalise_hyp_token(raw)
                        if ph not in skip:
                            hyp_seq.append(ph)
                prev = tok

        S, D, I = edit_distance(ref_seq, hyp_seq)
        R       = len(ref_seq)
        per_file[fname] = {"S": S, "D": D, "I": I, "R": R, "PER": (S+D+I)/R}
        total_S += S; total_D += D; total_I += I; total_ref += R

    overall_per = (total_S + total_D + total_I) / max(total_ref, 1)

    print(f"\n{'='*54}")
    print(f"  PHONEME ERROR RATE  (VAD chunking)")
    print(f"{'='*54}")
    print(f"  PER           : {overall_per*100:.2f}%")
    print(f"  Substitutions : {total_S:>6}  ({total_S/total_ref*100:.1f}%)")
    print(f"  Deletions     : {total_D:>6}  ({total_D/total_ref*100:.1f}%)")
    print(f"  Insertions    : {total_I:>6}  ({total_I/total_ref*100:.1f}%)")
    print(f"  Ref phonemes  : {total_ref:>6}")
    print(f"{'='*54}")

    print(f"\n  Worst files by PER:")
    for fn, m in sorted(per_file.items(), key=lambda x: -x[1]["PER"])[:10]:
        print(f"  {fn:<40} PER={m['PER']*100:5.1f}%  "
              f"R={m['R']}  S={m['S']}  D={m['D']}  I={m['I']}")

    return overall_per, per_file


overall_per, per_file = compute_per(results, ctc_tokenizer)



  PHONEME ERROR RATE  (VAD chunking)
  PER           : 13.51%
  Substitutions :   4137  (4.6%)
  Deletions     :   5309  (5.9%)
  Insertions    :   2677  (3.0%)
  Ref phonemes  :  89750

  Worst files by PER:
  Rhap-D0009                               PER= 29.3%  R=2999  S=209  D=546  I=123
  Rhap-D0006                               PER= 26.2%  R=3447  S=128  D=645  I=131
  Rhap-D2003                               PER= 22.5%  R=3935  S=264  D=534  I=86
  Rhap-D2007                               PER= 22.1%  R=2524  S=201  D=257  I=100
  Rhap-D0005                               PER= 22.1%  R=2398  S=117  D=342  I=71
  Rhap-D0002                               PER= 20.1%  R=2727  S=136  D=319  I=94
  Rhap-M0023                               PER= 18.0%  R=734  S=59  D=49  I=24
  Rhap-M1001                               PER= 17.3%  R=982  S=89  D=30  I=51
  Rhap-D2010                               PER= 17.0%  R=2823  S=137  D=243  I=99
  Rhap-D0008                               PER= 16.7%  

TypeError: compute_per() missing 1 required positional argument: 'ctc_tokenizer'

In [16]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def visualize_alignment(entry, fname, out_dir="results/alignment_plots", max_phones=80):
    """
    Plots the alignment matrix A for one file.
    If the file has multiple chunks, plots each chunk separately.
    
    Each plot shows:
      - Top:    alignment matrix A (N phonemes x T frames) as heatmap
      - Bottom: column argmax path (which phoneme dominates each frame)
                overlaid with the ideal diagonal
    """
    os.makedirs(out_dir, exist_ok=True)
    
    ref = entry["ref_intervals"]
    hyp = entry["hyp_intervals"]
    a_chunks = entry.get("A_chunks", [])
    
    if not a_chunks:
        print(f"  No A matrix stored for {fname}")
        return

    for chunk_idx, A in enumerate(a_chunks):
        A_np = A.numpy()          # (N, T)
        N, T = A_np.shape

        # Trim to max_phones for readability
        N_plot = min(N, max_phones)
        A_plot = A_np[:N_plot, :]

        # Phoneme labels for y-axis (from hyp_intervals for this chunk)
        # Approximate which hyp intervals belong to this chunk
        ph_labels = [iv["phoneme"] for iv in hyp][:N_plot]

        fig = plt.figure(figsize=(min(T * 0.05, 24), 6))
        gs  = gridspec.GridSpec(2, 1, height_ratios=[4, 1], hspace=0.05)

        # ── Top: heatmap ─────────────────────────────────────────────
        ax_heat = fig.add_subplot(gs[0])
        ax_heat.imshow(
            A_plot,
            aspect="auto",
            origin="upper",
            cmap="Blues",
            interpolation="nearest",
        )
        ax_heat.set_ylabel("Phoneme")
        ax_heat.set_xticks([])
        if ph_labels:
            ax_heat.set_yticks(range(N_plot))
            ax_heat.set_yticklabels(ph_labels, fontsize=6)
        ax_heat.set_title(
            f"{fname}  chunk {chunk_idx+1}/{len(a_chunks)}  "
            f"(N={N}, T={T})",
            fontsize=9,
        )

        # Ideal diagonal
        diag_t = np.linspace(0, T - 1, N_plot)
        ax_heat.plot(diag_t, np.arange(N_plot),
                     "r--", lw=1.0, alpha=0.6, label="ideal diagonal")
        ax_heat.legend(fontsize=7, loc="upper left")

        # ── Bottom: argmax path ───────────────────────────────────────
        ax_path = fig.add_subplot(gs[1])
        col_argmax = A_np.argmax(axis=0)          # (T,)
        # After computing col_argmax, apply running-max and overlay
        col_argmax_raw = A_np.argmax(axis=0).copy()
        col_argmax_mono = col_argmax_raw.copy()
        for t in range(1, T):
            if col_argmax_mono[t] < col_argmax_mono[t-1]:
                col_argmax_mono[t] = col_argmax_mono[t-1]
        
        ax_path.plot(col_argmax_raw,  color="steelblue", lw=0.8, alpha=0.6, label="raw argmax")
        ax_path.plot(col_argmax_mono, color="darkorange", lw=1.2, label="monotonic (used)")
        ax_path.legend(fontsize=7)
        ax_path.plot(col_argmax, color="steelblue", lw=0.8, label="argmax path")

        # Ideal diagonal in path space
        ideal = np.linspace(0, N - 1, T)
        ax_path.plot(ideal, color="red", lw=0.8, linestyle="--", alpha=0.6,
                     label="ideal")
        ax_path.set_xlim(0, T)
        ax_path.set_ylim(0, N)
        ax_path.set_xlabel("Frame (20ms)")
        ax_path.set_ylabel("Ph idx")
        ax_path.legend(fontsize=7, loc="upper left")

        plt.tight_layout()
        save_path = os.path.join(
            out_dir, f"{fname}_chunk{chunk_idx+1:02d}.png"
        )
        fig.savefig(save_path, dpi=120, bbox_inches="tight")
        plt.close()
        print(f"  Saved: {save_path}")

In [22]:
import soundfile as sf

# Save and listen to the first half of chunk 4 (the stuck region)
stuck_audio = chunk_audio[:int(2.0 * 16000)]   # first 2 seconds
sf.write("chunk4_stuck.wav", stuck_audio, 16000)

# And the second half (the compressed region)  
compressed_audio = chunk_audio[int(2.0*16000):int(2.3*16000)]  # frames 100-115
sf.write("chunk4_compressed.wav", compressed_audio, 16000)

In [18]:
for fname, entry in results.items():
    if fname == "Rhap-D2004":
        continue
    visualize_alignment(entry, fname)
    

/tmp/ipykernel_2147298/3314830485.py:90: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  Saved: results/alignment_plots/Rhap-M0012_chunk01.png


/tmp/ipykernel_2147298/3314830485.py:90: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  Saved: results/alignment_plots/Rhap-M0015_chunk01.png


/tmp/ipykernel_2147298/3314830485.py:90: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


...

  Saved: results/alignment_plots/Rhap-D1001_chunk07.png
  Saved: results/alignment_plots/Rhap-D1001_chunk08.png


/tmp/ipykernel_2147298/3314830485.py:90: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_2147298/3314830485.py:90: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  Saved: results/alignment_plots/Rhap-D1001_chunk09.png


/tmp/ipykernel_2147298/3314830485.py:90: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


KeyboardInterrupt: 

In [23]:
def plot_prior_dominance(N, T, align_temp=15.0, gaussian_w=30.0, ahead_w=40.0):
    """
    Shows how long phoneme 0 dominates the softmax based on prior alone
    (assuming uniform cosine similarity = 0 for all phonemes).
    """
    n_idx = np.arange(N) / max(N - 1, 1)
    t_idx = np.arange(T) / max(T - 1, 1)
    diff  = n_idx[:, None] - t_idx[None, :]          # (N, T)

    prior  = -gaussian_w * diff**2
    prior -= ahead_w * np.clip(diff, 0, None)**2      # (N, T)

    # Softmax over phonemes with prior only (cosine sim = 0)
    A_prior = np.exp(prior) / np.exp(prior).sum(axis=0, keepdims=True)

    fig, axes = plt.subplots(2, 1, figsize=(12, 5), height_ratios=[4, 1])
    axes[0].imshow(A_prior, aspect="auto", origin="upper",
                   cmap="Blues", interpolation="nearest")
    axes[0].set_title(f"Prior-only softmax (N={N}, T={T}) — no cosine similarity")
    axes[0].set_ylabel("Phoneme")
    axes[0].set_xlabel("Frame")

    axes[1].plot(A_prior.argmax(axis=0), color="steelblue", lw=1)
    axes[1].plot(np.linspace(0, N-1, T), "r--", lw=1, label="ideal")
    axes[1].set_ylabel("Argmax ph")
    axes[1].set_xlabel("Frame")
    axes[1].legend()
    plt.tight_layout()
    plt.savefig("results/prior_dominance.png", dpi=150)
    plt.close()
    print("Saved: results/prior_dominance.png")

# Run with your typical chunk dimensions

plot_prior_dominance(N=17, T=151)

Saved: results/prior_dominance.png
Saved: results/prior_dominance.png


In [24]:
plot_prior_dominance(N=21, T=210)

Saved: results/prior_dominance.png


In [25]:
def plot_cosine_similarity(entry, chunk_idx=0):
    """
    Plots the raw cosine similarity matrix D (before adding prior)
    so you can see what the model actually 'sees' acoustically.
    """
    A_chunks = entry.get("A_chunks", [])
    if not A_chunks:
        return

    # You need to re-run the model and capture D_mat before softmax
    # Add this temporarily to compute_alignment():
    #   self.last_D_mat = D_mat.clone()  (before adding prior)
    # Then access model.last_D_mat here
    pass


def plot_D_vs_prior(model, chunk_audio, feature_extractor, ctc_tokenizer, device):
    """
    Runs one chunk and separates cosine similarity from prior contribution.
    """
    inputs = feature_extractor(
        chunk_audio, sampling_rate=16000,
        return_tensors="pt", return_attention_mask=True,
    )
    with torch.no_grad():
        outputs = model.wav2vec2(
            inputs.input_values.to(device),
            inputs.attention_mask.to(device),
        )
        X      = outputs.last_hidden_state
        logits = model.ctc_head(X)
        labels_clean = model._ctc_decode_batch(logits)
        Y_emb  = model.compute_phoneme_embeddings(labels_clean, device=X.device)

        # Cosine similarity only (no prior, no temperature)
        X_proj = model.fx(X)
        X_norm = F.normalize(X_proj, dim=-1)
        Y_norm = F.normalize(Y_emb,  dim=-1)
        D_raw  = torch.matmul(Y_norm, X_norm.transpose(1, 2))[0].cpu().numpy()  # (N, T)

        # Prior only
        N, T   = D_raw.shape
        n_idx  = np.arange(N) / max(N-1, 1)
        t_idx  = np.arange(T) / max(T-1, 1)
        diff   = n_idx[:, None] - t_idx[None, :]
        prior  = -30.0 * diff**2 - 40.0 * np.clip(diff, 0, None)**2

    fig, axes = plt.subplots(3, 1, figsize=(14, 9),
                             gridspec_kw={"height_ratios": [3, 3, 3]})

    axes[0].imshow(D_raw, aspect="auto", origin="upper",
                   cmap="RdBu_r", interpolation="nearest")
    axes[0].set_title("Cosine similarity only  (no prior, no temperature)")
    axes[0].set_ylabel("Phoneme")

    axes[1].imshow(prior, aspect="auto", origin="upper",
                   cmap="RdBu_r", interpolation="nearest")
    axes[1].set_title("Prior only")
    axes[1].set_ylabel("Phoneme")

    # Full score = temperature * cosine + prior
    full = 15.0 * D_raw + prior
    A_full = np.exp(full) / np.exp(full).sum(axis=0, keepdims=True)
    axes[2].imshow(A_full, aspect="auto", origin="upper",
                   cmap="Blues", interpolation="nearest")
    axes[2].set_title("Final A = softmax(15 × cosine + prior)")
    axes[2].set_ylabel("Phoneme")
    axes[2].set_xlabel("Frame")

    plt.tight_layout()
    plt.savefig("results/D_vs_prior.png", dpi=150)
    plt.close()
    print("Saved: results/D_vs_prior.png")

In [28]:
c = chunks[3]
chunk_audio = audio[int(c['start']*16000):int(c['end']*16000)]
plot_D_vs_prior(model, chunk_audio, feature_extractor, ctc_tokenizer, device)

Saved: results/D_vs_prior.png


In [23]:

# =============================================================================
# Diagnostics
# =============================================================================

THRESHOLDS    = [0.010, 0.020, 0.050]
THRESHOLDS_MS = [10,    20,    50]


# -- 1. Boundary accuracy -----------------------------------------------------

def boundary_accuracy(results, thresholds=THRESHOLDS):
    errors  = []
    correct = {t: 0 for t in thresholds}
    total   = 0
    for entry in results.values():
        ref = entry["ref_intervals"]
        hyp = entry["hyp_intervals"]
        if len(ref) < 2 or len(hyp) < 2:
            continue
        ref_bounds = [iv["end"] for iv in ref[:-1]]
        hyp_bounds = sorted(
            {iv["start"] for iv in hyp} | {iv["end"] for iv in hyp}
        )
        for rb in ref_bounds:
            nearest = min(hyp_bounds, key=lambda h: abs(h - rb))
            err = nearest - rb
            errors.append(err)
            total += 1
            for t in thresholds:
                if abs(err) <= t:
                    correct[t] += 1
    acc = {t: correct[t] / max(total, 1) for t in thresholds}
    return acc, np.array(errors), total


acc, errors, n_bounds = boundary_accuracy(results)

print(f"\n{'='*54}")
print(f"  BOUNDARY ACCURACY  ({n_bounds} boundaries)")
print(f"{'='*54}")
for t, ms in zip(THRESHOLDS, THRESHOLDS_MS):
    bar = "X" * int(acc[t] * 40)
    print(f"  +/-{ms:>3}ms : {acc[t]*100:5.2f}%  {bar}")
print(f"  MAE    : {np.mean(np.abs(errors))*1000:.1f} ms")
print(f"  Bias   : {np.mean(errors)*1000:+.1f} ms  (+ = hyp runs ahead)")
print(f"  Median : {np.median(errors)*1000:+.1f} ms")
print(f"{'='*54}\n")


# -- 2. Boundary error histogram -----------------------------------------------

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(errors * 1000, bins=120, range=(-250, 250),
        color="steelblue", edgecolor="none")
ax.axvline(0, color="red", lw=1.5, linestyle="--", label="perfect")
ax.axvline(np.mean(errors) * 1000, color="orange", lw=1.5,
           label=f"bias {np.mean(errors)*1000:+.1f} ms")
for ms in THRESHOLDS_MS:
    ax.axvline( ms, color="green", lw=0.7, linestyle=":")
    ax.axvline(-ms, color="green", lw=0.7, linestyle=":")
ax.set_xlabel("Boundary error (ms)  [hyp - ref]")
ax.set_ylabel("Count")
ax.set_title("Boundary error distribution  (bertphone)")
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig("results/boundary_error_hist.png", dpi=150)
plt.close()
print("Saved: results/boundary_error_hist.png")


# -- 3. Positional bias --------------------------------------------------------

def positional_bias(results, n_bins=10):
    bin_errors = defaultdict(list)
    for entry in results.values():
        ref = entry["ref_intervals"]
        hyp = entry["hyp_intervals"]
        if len(ref) < 4:
            continue
        dur = ref[-1]["end"] - ref[0]["start"]
        if dur < 0.5:
            continue
        hyp_bounds = sorted(
            {iv["start"] for iv in hyp} | {iv["end"] for iv in hyp}
        )
        for iv in ref[:-1]:
            rb  = iv["end"]
            pos = (rb - ref[0]["start"]) / dur
            nearest = min(hyp_bounds, key=lambda h: abs(h - rb))
            bin_errors[min(int(pos * n_bins), n_bins - 1)].append(nearest - rb)

    centres = [(k + 0.5) / n_bins for k in range(n_bins)]
    means   = [np.mean(bin_errors[k]) * 1000 if bin_errors[k] else 0.0
               for k in range(n_bins)]
    stds    = [np.std(bin_errors[k])  * 1000 if bin_errors[k] else 0.0
               for k in range(n_bins)]
    return centres, means, stds


centres, means, stds = positional_bias(results)
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(centres, means, width=0.08,
       color=["tomato" if m > 0 else "steelblue" for m in means], alpha=0.85)
ax.errorbar(centres, means, yerr=stds,
            fmt="none", color="black", capsize=3, lw=1)
ax.axhline(0, color="black", lw=1)
ax.set_xlabel("Relative position in utterance  (0 = start, 1 = end)")
ax.set_ylabel("Mean boundary error (ms)")
ax.set_title("Positional bias -- running-ahead diagnosis")
ax.set_xticks([i / 10 for i in range(11)])
plt.tight_layout()
fig.savefig("results/positional_bias.png", dpi=150)
plt.close()
print("Saved: results/positional_bias.png")


  BOUNDARY ACCURACY  (89697 boundaries)
  +/- 10ms : 18.86%  XXXXXXX
  +/- 20ms : 33.93%  XXXXXXXXXXXXX
  +/- 50ms : 73.35%  XXXXXXXXXXXXXXXXXXXXXXXXXXXXX
  MAE    : 90.2 ms
  Bias   : -1.6 ms  (+ = hyp runs ahead)
  Median : +0.3 ms

Saved: results/boundary_error_hist.png
Saved: results/positional_bias.png


In [28]:
def alignment_diagnostics(results):
    sharpness_list = []
    diagonal_list  = []
    entropy_list   = []
    ahead_list     = []
    n_chunks_total = 0

    for entry in results.values():
        for A in entry.get("A_chunks", []):   # ← iterate over stored chunks
            A_np = A.numpy() if hasattr(A, "numpy") else np.array(A)
            N, T = A_np.shape
            if N < 2 or T < 2:
                continue

            col_argmax = A_np.argmax(axis=0)
            col_max    = A_np.max(axis=0)
            sharpness_list.append((col_max > 1.0 / N).mean())

            t_norm = np.arange(T) / max(T - 1, 1)
            n_norm = col_argmax / max(N - 1, 1)
            diagonal_list.append((np.abs(n_norm - t_norm) < 0.15).mean())
            ahead_list.append((n_norm < t_norm - 0.10).mean())

            eps = 1e-9
            entropy_list.append(
                -(A_np * np.log(A_np + eps)).sum(axis=0).mean()
            )
            n_chunks_total += 1

    if not sharpness_list:
        print("No A_chunks stored.")
        return

    print(f"\n{'='*54}")
    print(f"  ALIGNMENT MATRIX DIAGNOSTICS  "
          f"({n_chunks_total} chunks, {len(results)} files)")
    print(f"{'='*54}")
    print(f"  Sharpness (col-max > 1/N)   : {np.mean(sharpness_list)*100:.1f}%")
    print(f"  Diagonal coverage (+/-15%)  : {np.mean(diagonal_list)*100:.1f}%")
    print(f"  Running-ahead fraction      : {np.mean(ahead_list)*100:.1f}%")
    print(f"  Mean column entropy (nats)  : {np.mean(entropy_list):.3f}")
    print(f"{'='*54}\n")

    # Per-file running-ahead distribution
    per_file_ahead = []
    for entry in results.values():
        chunks = entry.get("A_chunks", [])
        if chunks:
            file_ahead = np.mean([
                (a.numpy().argmax(axis=0) / max(a.shape[0]-1, 1)
                 < np.arange(a.shape[1]) / max(a.shape[1]-1, 1) - 0.10).mean()
                for a in chunks
            ])
            per_file_ahead.append(file_ahead)

    fig, axes = plt.subplots(1, 3, figsize=(13, 3))

    axes[0].hist(np.array(sharpness_list)*100, bins=30,
                 color="steelblue", edgecolor="none")
    axes[0].set_xlabel("Sharpness (%)")
    axes[0].set_title("Column-max sharpness")

    axes[1].hist(np.array(ahead_list)*100, bins=30,
                 color="tomato", edgecolor="none")
    axes[1].set_xlabel("Frames ahead of diagonal (%)")
    axes[1].set_title("Running-ahead per chunk")

    axes[2].hist(np.array(entropy_list), bins=30,
                 color="goldenrod", edgecolor="none")
    axes[2].set_xlabel("Column entropy (nats)")
    axes[2].set_title("Attention entropy per chunk")

    plt.tight_layout()
    fig.savefig("results/alignment_quality.png", dpi=150)
    plt.close()
    print("Saved: results/alignment_quality.png")

In [29]:
alignment_diagnostics(results)


  ALIGNMENT MATRIX DIAGNOSTICS  (105 chunks, 53 files)
  Sharpness (col-max > 1/N)   : 100.0%
  Diagonal coverage (+/-15%)  : 78.0%
  Running-ahead fraction      : 26.8%
  Mean column entropy (nats)  : 3.153

Saved: results/alignment_quality.png


In [30]:
# ── Debug 1: per-file MAE distribution ───────────────────────────────
# If a subset of files has MAE > 200ms, they are the source of the damage.

file_maes = {}
for fname, entry in results.items():
    ref = entry["ref_intervals"]
    hyp = entry["hyp_intervals"]
    if len(ref) < 2 or len(hyp) < 2:
        continue
    hyp_bounds = sorted(
        {iv["start"] for iv in hyp} | {iv["end"] for iv in hyp}
    )
    ref_bounds = [iv["end"] for iv in ref[:-1]]
    errs = [min(abs(rb - h) for h in hyp_bounds) for rb in ref_bounds]
    file_maes[fname] = np.mean(errs) * 1000

maes = np.array(list(file_maes.values()))
print(f"\nPer-file MAE distribution:")
for pct in [25, 50, 75, 90, 95, 99]:
    print(f"  p{pct:>2} : {np.percentile(maes, pct):.1f} ms")

bad_files = [(f, m) for f, m in file_maes.items() if m > 200]
print(f"\nFiles with MAE > 200ms: {len(bad_files)} / {len(file_maes)}")
for fname, mae in sorted(bad_files, key=lambda x: -x[1])[:10]:
    entry  = results[fname]
    ref    = entry["ref_intervals"]
    hyp    = entry["hyp_intervals"]
    print(f"  {fname:<40}  MAE={mae:6.1f}ms  "
          f"ref_n={len(ref):4d}  hyp_n={len(hyp):4d}  "
          f"ref_end={ref[-1]['end'] if ref else 0:.2f}s  "
          f"hyp_end={hyp[-1]['end'] if hyp else 0:.2f}s")


Per-file MAE distribution:
  p25 : 62.6 ms
  p50 : 91.1 ms
  p75 : 125.9 ms
  p90 : 172.2 ms
  p95 : 181.5 ms
  p99 : 198.9 ms

Files with MAE > 200ms: 1 / 53
  Rhap-M0021                                MAE= 211.0ms  ref_n= 546  hyp_n= 476  ref_end=86.47s  hyp_end=86.24s


In [31]:
# ── Debug 2: count mismatch ───────────────────────────────────────────
# Large ref_n vs hyp_n gap on a bad file = wrong pairing or offset uncaught.

print(f"\nRef / Hyp count ratio (all files):")
ratios = []
for entry in results.values():
    r, h = len(entry["ref_intervals"]), len(entry["hyp_intervals"])
    if h > 0:
        ratios.append(r / h)
ratios = np.array(ratios)
print(f"  Mean  : {ratios.mean():.2f}")
print(f"  Median: {np.median(ratios):.2f}")
print(f"  Files with ratio > 2x: {(ratios > 2).sum()}")
print(f"  Files with ratio < 0.5x: {(ratios < 0.5).sum()}")


Ref / Hyp count ratio (all files):
  Mean  : 1.13
  Median: 1.12
  Files with ratio > 2x: 0
  Files with ratio < 0.5x: 0


In [32]:

# -- 5. Per-phoneme boundary accuracy -----------------------------------------

def per_phoneme_accuracy(results, threshold=0.020):
    ph_correct = defaultdict(int)
    ph_total   = defaultdict(int)
    for entry in results.values():
        ref = entry["ref_intervals"]
        hyp = entry["hyp_intervals"]
        if len(ref) < 2 or len(hyp) < 2:
            continue
        hyp_bounds = sorted(
            {iv["start"] for iv in hyp} | {iv["end"] for iv in hyp}
        )
        for iv in ref[:-1]:
            rb  = iv["end"]
            ph  = iv["phoneme"]
            nearest = min(hyp_bounds, key=lambda h: abs(h - rb))
            ph_total[ph] += 1
            if abs(nearest - rb) <= threshold:
                ph_correct[ph] += 1

    rows = sorted(
        [(ph, ph_correct[ph] / tot, tot)
         for ph, tot in ph_total.items()],
        key=lambda r: -r[2],
    )
    ms = int(threshold * 1000)
    print(f"\n{'='*54}")
    print(f"  PER-PHONEME BOUNDARY ACCURACY  @ +/-{ms}ms")
    print(f"{'='*54}")
    print(f"  {'Ph':<8} {'Acc':>6}  {'N':>5}  bar")
    for ph, a, tot in rows:
        bar = "|" * int(a * 25)
        print(f"  {str(ph):<8} {a*100:5.1f}%  {tot:>5}  {bar}")
    print(f"{'='*54}\n")
    return rows


per_phoneme_accuracy(results)





  PER-PHONEME BOUNDARY ACCURACY  @ +/-20ms
  Ph          Acc      N  bar
  a         32.9%   7654  ||||||||
  ʁ         36.1%   6631  |||||||||
  e         32.4%   5578  ||||||||
  l         35.2%   5555  ||||||||
  s         34.9%   5253  ||||||||
  i         33.8%   4937  ||||||||
  t         35.0%   4566  ||||||||
  d         34.7%   3930  ||||||||
  ɛ         34.6%   3913  ||||||||
  k         33.9%   3808  ||||||||
  ə         30.9%   3302  |||||||
  p         36.5%   3267  |||||||||
  @         31.9%   3004  |||||||
  m         34.6%   2906  ||||||||
  n         33.3%   2417  ||||||||
  v         35.6%   2407  ||||||||
  §         30.3%   1950  |||||||
  u         32.4%   1912  ||||||||
  ɔ         35.3%   1807  ||||||||
  y         32.8%   1787  ||||||||
  j         33.8%   1615  ||||||||
  ʒ         33.0%   1487  ||||||||
  z         38.6%   1275  |||||||||
  f         34.0%   1243  ||||||||
  o         31.0%   1148  |||||||
  b         32.8%   1087  ||||||||
  w         35.9%

[('a', 0.3293702639142932, 7654),
 ('ʁ', 0.36103151862464183, 6631),
 ('e', 0.32413051272857657, 5578),
 ('l', 0.35175517551755175, 5555),
 ('s', 0.34875309347039785, 5253),
 ('i', 0.338464654648572, 4937),
 ('t', 0.3495400788436268, 4566),
 ('d', 0.3470737913486005, 3930),
 ('ɛ', 0.34551495016611294, 3913),
 ('k', 0.3390231092436975, 3808),
 ('ə', 0.30920654149000604, 3302),
 ('p', 0.3654729109274564, 3267),
 ('@', 0.31857523302263646, 3004),
 ('m', 0.346180316586373, 2906),
 ('n', 0.33347124534546957, 2417),
 ('v', 0.35646032405484007, 2407),
 ('§', 0.30256410256410254, 1950),
 ('u', 0.323744769874477, 1912),
 ('ɔ', 0.35251798561151076, 1807),
 ('y', 0.3279238947957471, 1787),
 ('j', 0.3380804953560371, 1615),
 ('ʒ', 0.3301950235373235, 1487),
 ('z', 0.38588235294117645, 1275),
 ('f', 0.3403057119871279, 1243),
 ('o', 0.31010452961672474, 1148),
 ('b', 0.32842686292548295, 1087),
 ('w', 0.35880708294501396, 1073),
 ('ø', 0.2962138084632517, 898),
 ('9', 0.2728442728442728, 777),
 ('1

In [17]:
# -- 6. Duration vs boundary accuracy -----------------------------------------

def duration_vs_error(results, threshold=0.020, n_bins=8):
    all_durs, all_errs = [], []
    for entry in results.values():
        ref = entry["ref_intervals"]
        hyp = entry["hyp_intervals"]
        if len(ref) < 2 or len(hyp) < 2:
            continue
        hyp_bounds = sorted(
            {iv["start"] for iv in hyp} | {iv["end"] for iv in hyp}
        )
        for iv in ref[:-1]:
            rb  = iv["end"]
            nearest = min(hyp_bounds, key=lambda h: abs(h - rb))
            all_durs.append(iv["end"] - iv["start"])
            all_errs.append(abs(nearest - rb))

    if not all_durs:
        return

    all_durs = np.array(all_durs)
    all_errs = np.array(all_errs)
    edges    = np.percentile(all_durs, np.linspace(0, 100, n_bins + 1))
    bin_acc, bin_cen = [], []
    for i in range(n_bins):
        mask = (all_durs >= edges[i]) & (all_durs <= edges[i + 1])
        if mask.sum() > 0:
            bin_acc.append((all_errs[mask] <= threshold).mean() * 100)
            bin_cen.append(((edges[i] + edges[i + 1]) / 2) * 1000)

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(bin_cen, bin_acc, "o-", color="steelblue", lw=2)
    ax.set_xlabel("Phoneme duration (ms)")
    ax.set_ylabel(f"Boundary acc @ +/-{int(threshold*1000)}ms (%)")
    ax.set_title("Boundary accuracy vs. phoneme duration")
    ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig("results/duration_vs_accuracy.png", dpi=150)
    plt.close()
    print("Saved: results/duration_vs_accuracy.png")


duration_vs_error(results)

print("\nAll diagnostics complete.")
for fname in [
    output,
    "results/boundary_error_hist.png",
    "results/positional_bias.png",
    "results/alignment_quality.png",
    "results/duration_vs_accuracy.png",
]:
    print(f"  {fname}")

Saved: results/duration_vs_accuracy.png

All diagnostics complete.
  results/rhap_bertphone.pkl
  results/boundary_error_hist.png
  results/positional_bias.png
  results/alignment_quality.png
  results/duration_vs_accuracy.png
